In [1]:
# ---------------------------------------------------------
# Step 1: Faithfulness evaluation setup
# ---------------------------------------------------------

from pathlib import Path
import json
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
)

EVALUATION_DIR = (
    RESULTS_DIR
    / "evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [10]:
# ---------------------------------------------------------
# Input paths
# ---------------------------------------------------------

CLINICAL_NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

STUDY_PATIENT_IDS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "patients"
    / "study_patient_ids.json"
)

In [4]:

# ---------------------------------------------------------
# Faithfulness output paths
# ---------------------------------------------------------

FAITHFULNESS_CLAIMS_PATH = (
    EVALUATION_DIR
    / "faithfulness_atomic_claims.json"
)

FAITHFULNESS_RESULTS_PATH = (
    EVALUATION_DIR
    / "faithfulness_results.json"
)

In [7]:
from openai import OpenAI

client = OpenAI()

In [8]:
# ---------------------------------------------------------
# Evaluator model
# ---------------------------------------------------------

EVALUATOR_MODEL = "gpt-5.4-mini"


print("Project root:", PROJECT_ROOT)
print("Evaluation directory:", EVALUATION_DIR)
print("Evaluator model:", EVALUATOR_MODEL)

Project root: /Users/pallavi_chandanshive/projects/clinical-summarization-eval
Evaluation directory: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation
Evaluator model: gpt-5.4-mini


In [11]:
# ---------------------------------------------------------
# Step 2: Load frozen cohort and four workflow summaries
# ---------------------------------------------------------

# Load the exact frozen 50-patient cohort.
with open(STUDY_PATIENT_IDS_PATH, "r") as f:
    study_patient_ids = json.load(f)

study_patient_id_set = set(study_patient_ids)

print("Study patients:", len(study_patient_ids))

Study patients: 50


In [12]:
# ---------------------------------------------------------
# Final workflow result paths
# ---------------------------------------------------------

workflow_paths = {
    "direct": (
        RESULTS_DIR
        / "direct"
        / "direct_summaries.json"
    ),
    "hierarchical": (
        RESULTS_DIR
        / "hierarchical"
        / "hierarchical_summaries.json"
    ),
    "rag": (
        RESULTS_DIR
        / "rag"
        / "final_rag_summaries.json"
    ),
    "rag_verification": (
        RESULTS_DIR
        / "rag_verification"
        / "final_verified_summaries.json"
    ),
}

In [13]:

# Each workflow stores its final summary under a different field.
summary_fields = {
    "direct": "summary",
    "hierarchical": "final_summary",
    "rag": "summary",
    "rag_verification": "final_summary",
}


In [14]:
# ---------------------------------------------------------
# Load summaries
# ---------------------------------------------------------

workflow_summaries = {}

for workflow, path in workflow_paths.items():

    with open(path, "r") as f:
        results = json.load(f)

    summary_field = summary_fields[workflow]

    workflow_summaries[workflow] = {
        person_id: data[summary_field]
        for person_id, data in results.items()
    }


In [15]:
# ---------------------------------------------------------
# Validate exact cohort for every workflow
# ---------------------------------------------------------

for workflow, summaries in workflow_summaries.items():

    workflow_ids = set(summaries.keys())

    print(
        workflow,
        "| summaries:",
        len(summaries),
        "| IDs match:",
        workflow_ids == study_patient_id_set
    )

direct | summaries: 50 | IDs match: True
hierarchical | summaries: 50 | IDs match: True
rag | summaries: 50 | IDs match: True
rag_verification | summaries: 50 | IDs match: True


In [16]:
# ---------------------------------------------------------
# Step 3: Reconstruct complete original patient records
# ---------------------------------------------------------

# Load original clinical notes.
notes = pd.read_csv(CLINICAL_NOTES_PATH)


# ---------------------------------------------------------
# Apply the exact frozen preprocessing used in the study
# ---------------------------------------------------------

# Remove notes whose entire cleaned text is "#NAME?".
notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()


# Sort chronologically and remove duplicate notes.
# If the same note text occurs more than once for a patient,
# retain only its earliest occurrence.
notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)


# Keep only the frozen 50-patient study cohort.
notes_dedup = notes_dedup[
    notes_dedup["person_id"].isin(study_patient_id_set)
].copy()


# ---------------------------------------------------------
# Build one complete source document per patient
# ---------------------------------------------------------

patient_source_documents = (
    notes_dedup
    .groupby("person_id", sort=False)["clean_note_text"]
    .apply(
        lambda texts: "\n\n".join(
            texts.astype(str)
        )
    )
    .to_dict()
)


# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

source_patient_ids = set(patient_source_documents.keys())

print("Source documents:", len(patient_source_documents))
print("Study patients:", len(study_patient_ids))
print(
    "IDs match:",
    source_patient_ids == study_patient_id_set
)

print(
    "Missing IDs:",
    study_patient_id_set - source_patient_ids
)

print(
    "Extra IDs:",
    source_patient_ids - study_patient_id_set
)

Source documents: 50
Study patients: 50
IDs match: True
Missing IDs: set()
Extra IDs: set()


In [17]:
# ---------------------------------------------------------
# Step 4A: Atomic claim extraction prompt
# ---------------------------------------------------------

FAITHFULNESS_CLAIM_SYSTEM_PROMPT = """
You are extracting atomic factual claims from a generated clinical summary
for a faithfulness evaluation.

Your task is ONLY to decompose the summary into independently verifiable
clinical claims.

An atomic claim should contain one factual proposition that can be checked
against the patient's original clinical record.

IMPORTANT RULES:

1. Preserve the exact clinical meaning of the generated summary.
2. Do NOT add, infer, correct, or reinterpret information.
3. Do NOT use outside medical knowledge.
4. Preserve important relationships between entities.

   For example, if a sentence states that medication A was given for
   condition B, the extracted claim must preserve that medication-condition
   relationship correctly.

5. When a modifier, indication, dose, timing, anatomical site, or other
   detail clearly belongs to a specific entity, keep it attached to that
   entity.

6. Split compound statements only when doing so does not change their
   meaning or create incorrect relationships.

7. Do not make a claim more specific than the original summary.
8. Do not make a claim more general than the original summary.
9. Include factual clinical statements such as diagnoses, symptoms,
   investigations, treatments, procedures, medication changes, clinical
   progression, and outcomes.
10. Exclude purely structural headings or non-factual filler.

Return ONLY valid JSON in this format:

{
  "claims": [
    {
      "claim_id": 1,
      "claim": "..."
    },
    {
      "claim_id": 2,
      "claim": "..."
    }
  ]
}
""".strip()

In [18]:
# ---------------------------------------------------------
# Step 4B: Select one stress-test summary
# ---------------------------------------------------------

PILOT_PERSON_ID = "05192757-942f-460d-b4ff-004ec39cc5ee"
PILOT_WORKFLOW = "rag"

pilot_summary = workflow_summaries[
    PILOT_WORKFLOW
][PILOT_PERSON_ID]

print("Patient:", PILOT_PERSON_ID)
print("Workflow:", PILOT_WORKFLOW)
print()
print(pilot_summary)

Patient: 05192757-942f-460d-b4ff-004ec39cc5ee
Workflow: rag

- **21/12/2025 – Preoperative assessment:** Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis, associated with stiffness and limited mobility. Medical history included hypertension, severe left hip OA, and cataract surgery in 2018. BP was 145/85 mmHg and Hb 11.3 g/dL, consistent with mild anaemia; she was taking ferrous sulfate 200 mg daily, paracetamol as needed, and calcium/vitamin D supplements. ECG showed normal sinus rhythm without ischaemic changes. No allergies were reported. She was deemed suitable for spinal anaesthesia, consented for elective total left hip replacement, and cleared for admission on 02/01/2026.

- **02/01/2026 – Admission and surgery:** Preoperative checks confirmed identity, fasting status, correct left hip site, medication reconciliation, and suitability to proceed. Mild anaemia had been optimised following ferrous sulfate, and elevated BP was c

In [19]:
# ---------------------------------------------------------
# Step 4C: Pilot atomic claim extraction
# ---------------------------------------------------------

pilot_user_prompt = f"""
GENERATED CLINICAL SUMMARY:

{pilot_summary}
""".strip()


response = client.chat.completions.create(
    model=EVALUATOR_MODEL,
    messages=[
        {
            "role": "system",
            "content": FAITHFULNESS_CLAIM_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": pilot_user_prompt,
        },
    ],
    response_format={"type": "json_object"},
)


# Parse the returned JSON.
pilot_claim_result = json.loads(
    response.choices[0].message.content
)

pilot_claims = pilot_claim_result["claims"]


print("Number of claims:", len(pilot_claims))
print()

for claim in pilot_claims:
    print(
        claim["claim_id"],
        "-",
        claim["claim"]
    )

Number of claims: 84

1 - On 21/12/2025, Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis.
2 - Her left hip pain was associated with stiffness and limited mobility.
3 - Her medical history included hypertension.
4 - Her medical history included severe left hip osteoarthritis.
5 - Her medical history included cataract surgery in 2018.
6 - At preoperative assessment, her blood pressure was 145/85 mmHg.
7 - At preoperative assessment, her haemoglobin was 11.3 g/dL.
8 - Her haemoglobin of 11.3 g/dL was consistent with mild anaemia.
9 - She was taking ferrous sulfate 200 mg daily.
10 - She was taking paracetamol as needed.
11 - She was taking calcium and vitamin D supplements.
12 - Her ECG showed normal sinus rhythm without ischaemic changes.
13 - No allergies were reported.
14 - She was deemed suitable for spinal anaesthesia.
15 - She consented to elective total left hip replacement.
16 - She was cleared for admission on 02/01/2026.
17

In [ ]:
# ---------------------------------------------------------
# Step 5: Extract atomic claims from all final summaries
# ---------------------------------------------------------

# Load existing progress if this cell is being resumed.
if FAITHFULNESS_CLAIMS_PATH.exists():

    with open(FAITHFULNESS_CLAIMS_PATH, "r") as f:
        faithfulness_atomic_claims = json.load(f)

    print("Loaded existing claim extraction progress.")

else:
    faithfulness_atomic_claims = {}


# ---------------------------------------------------------
# Extract claims for each patient × workflow
# ---------------------------------------------------------

for workflow, summaries in workflow_summaries.items():

    # Create workflow container if it does not already exist.
    if workflow not in faithfulness_atomic_claims:
        faithfulness_atomic_claims[workflow] = {}

    for person_id in study_patient_ids:

        # Skip summaries already processed.
        if person_id in faithfulness_atomic_claims[workflow]:
            continue

        summary = summaries[person_id]

        user_prompt = f"""
GENERATED CLINICAL SUMMARY:

{summary}
""".strip()

        response = client.chat.completions.create(
            model=EVALUATOR_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": FAITHFULNESS_CLAIM_SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
            ],
            response_format={"type": "json_object"},
        )

        result = json.loads(
            response.choices[0].message.content
        )

        claims = result["claims"]

        faithfulness_atomic_claims[workflow][person_id] = claims

        # Save after every completed summary so progress
        # survives notebook interruption.
        with open(FAITHFULNESS_CLAIMS_PATH, "w") as f:
            json.dump(
                faithfulness_atomic_claims,
                f,
                indent=2
            )

        print(
            workflow,
            person_id,
            "| claims:",
            len(claims)
        )


print("\nClaim extraction complete.")

direct 028998ee-babc-4096-9b28-001bc2f9a84e | claims: 53
direct 04df53ea-55c1-48d9-84a1-1f15c133b29b | claims: 59
direct 05192757-942f-460d-b4ff-004ec39cc5ee | claims: 69
direct 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf | claims: 86
direct 0f438665-d430-4adb-8acc-c3beed9e4942 | claims: 61
